# 05 · 부하 실험과 프로젝트 연결

성공 지연·오류율·성공 처리량을 같이 읽습니다. 권장 20분. 고정 응답·수면 모사이며 GPU 벤치마크가 아닙니다.

위에서 아래로 실행합니다. 각 실험은 새 앱을 사용하며 실제 LLM·GPU·외부 API는 기본 경로에 없습니다. 먼저 결과를 예상하고 실행한 뒤 변수 하나를 바꾸고 관찰을 기록하세요.

In [ ]:
from pathlib import Path
import sys, asyncio, json, time
from dataclasses import replace
import numpy as np
import httpx
from IPython.display import display
ROOT = next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'course_labs').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from week09.lab.app import create_app
from week09.lab.service import Settings, ChatRequest, Service, TTLCache, TokenBucket, cache_key
from week09.lab.benchmark import run_load, summarize
assert sys.version_info[:2] == (3,11), 'Python 3.11 .venv 커널 선택'
OUT = ROOT/'output/validation/week09-lab'
OUT.mkdir(parents=True,exist_ok=True)
print('Python',sys.version.split()[0],'NumPy',np.__version__,'HTTPX',httpx.__version__)


## 실험 조건

새 앱에서 캐시를 끄고 동일 작업을 수행합니다. warmup은 지표에서 제외합니다.

In [ ]:
cfg = Settings(max_inflight=2,mock_delay_s=.03,timeout_s=.3)
reports = []
for c in [1,2,6]:
    app = create_app(cfg)
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app),base_url='http://lab') as client:
        report = await run_load(client,n=24,concurrency=c,use_cache=False,warmup=2)
    reports.append({'concurrency':c,**report})
assert all(r['summary']['n']==24 for r in reports)
display([{'c':r['concurrency'],**r['summary']} for r in reports])

## 성공과 거절

동시 요청 수가 처리 용량보다 크면 503이 늘 수 있습니다. 낮은 지연을 개선으로만 읽지 않습니다.

In [ ]:
assert reports[0]['summary']['success'] == 24
assert reports[1]['summary']['success'] == 24
assert reports[2]['summary']['error_rate'] > 0
display([{'c':r['concurrency'],'errors':r['summary']['status_counts'],'goodput':r['summary']['successful_rps']} for r in reports])

## 분위수의 의미

수식용 합성 데이터로 계산을 확인합니다. 24개 요청의 p99를 안정된 운영 지표로 보지 않습니다.

In [ ]:
sample = [10,10,10,10,100]
assert np.percentile(sample,50) == 10
assert np.percentile(sample,95) > 80
display({'mean':np.mean(sample),'p50':np.percentile(sample,50),'p95':np.percentile(sample,95),'method':'linear interpolation'})

## 실패를 섞은 착시

빠른 거절이 전체 평균을 낮춥니다. 성공 지연 분포와 실패율을 함께 보고합니다.

In [ ]:
rows = [{'status':200,'elapsed_ms':100}]+[{'status':503,'elapsed_ms':1} for _ in range(9)]
s = summarize(rows,1)
assert s['error_rate'] == .9 and s['success_latency_ms']['p50'] == 100
display({'all_mean':np.mean([r['elapsed_ms'] for r in rows]),**s})

## 캐시 적중 부하의 분리

같은 질문을 먼저 캐시한 뒤 비교합니다. 모델을 호출하지 않은 결과를 모델 추론 속도라고 쓰지 않습니다.

In [ ]:
app = create_app(cfg)
async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app),base_url='http://lab') as client:
    cached = await run_load(client,n=24,concurrency=2,use_cache=True,warmup=2)
assert app.state.service.provider_calls == 1
display(cached['summary'])

## 실험 한계

폐쇄형 부하에서는 응답이 느려지면 다음 요청 유입도 줄어듭니다. 고정 도착률 시험과 구분합니다.

In [ ]:
limitations = ['ASGI 직접 호출: 실제 네트워크 제외','고정 수면: GPU·토큰 생성 제외','동시성 한도: 8, 요청 최대100','폐쇄형 부하: 도착률 고정 아님','표본24: 장시간 SLO 보장 아님']
display(limitations)

## 실제 제공자 어댑터 검사

외부 모델 없이 HTTPX MockTransport로 연결·모델명 오류를 확인합니다. 다음 셀은 실제 Ollama를 호출하지 않습니다.

In [ ]:
def upstream(request):
    if request.url.path == '/api/tags': return httpx.Response(200,json={'models':[]})
    return httpx.Response(404,json={'error':'synthetic missing model'})
app = create_app(Settings(provider='ollama',model='not-installed'),transport=httpx.MockTransport(upstream))
async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app),base_url='http://lab') as client:
    ready = await client.get('/ready')
    missing = await client.post('/api/v1/chat',json={'message':'x'})
assert ready.status_code == 503 and missing.json()['error']['code'] == 'model_not_found'
display(missing.json())

## 성공 조건과 실패 처리

요구사항을 API·시험·실패 동작에 연결합니다. 목표 시간은 제안값이지 측정 성과가 아닙니다.

In [ ]:
requirements = [{'id':'FR01','endpoint':'POST /api/v1/chat','test':'정상·공백·필드 오류','fallback':'입력 수정 안내'}, {'id':'FR02','endpoint':'/api/v1/chat/stream','test':'done·error·중단','fallback':'미완료 표시'}, {'id':'NFR01','endpoint':'부하 실험','test':'성공 p95+오류율+성공 처리량','fallback':'요청 제한·용량 조정'}]
assert all(r['test'] and r['fallback'] for r in requirements)
display(requirements)

## 실험 근거 저장

숫자는 실행 환경마다 달라집니다. 표본 수·정의·조건과 함께 저장합니다.

In [ ]:
report = {'scope':'local ASGI mock; not LLM or network benchmark','settings':{'delay_s':.03,'capacity':2,'requests':24,'cache':False},'runs':reports,'cached':cached,'requirements':requirements,'limitations':limitations}
(OUT/'benchmark-report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print(OUT/'benchmark-report.json')

## 기획서 이후의 작업

새 기획서 제출을 반복하지 않고 최소 기능·실패 복구·실험 증거를 구현 포트폴리오에 누적합니다.

In [ ]:
next_step = {'service':'API로 제공할 최소 기능 1개','proof':'성공·입력 오류·시간 초과·취소','decision':'유지/수정/보류와 근거','next_week':'배포 구조·관찰·복구'}
display(next_step)

## 포트폴리오 기록

변경 변수 / 고정 조건 / 실제 관측값 / 실패 사례 / 해석의 한계 / 다음 확인을 기록하세요. 개인정보·학생 원문은 공개 저장소에 넣지 않습니다. 별도 과제 마감은 소속 공지를 따르며 이 실습에서 새 제출 기한을 만들지 않습니다.